# 05. Multi-Kaggle dataset research

Ноутбук нужен, чтобы сразу проверить несколько Kaggle-датасетов и выбрать, на каком дальше запускать полный pipeline.

Что делает notebook:

1. скачивает несколько Kaggle datasets через `kagglehub.dataset_download`;
2. читает все таблицы внутри каждого датасета;
3. ищет date/time, text/title/body, asset/symbol/coin columns;
4. считает упоминания BTC/ETH/SOL/BNB/XRP и других монет;
5. подтягивает prices через `yfinance`;
6. оценивает, где лучше пересечение news + prices;
7. строит общий leaderboard датасетов;
8. сохраняет short-list для будущего RUN 05 pipeline.

Важно: этот ноутбук пока не запускает FinBERT. Сначала выбираем лучший датасет / актив / горизонт.

## 0. Установка пакетов

In [ ]:
%pip install -q kagglehub yfinance pandas numpy matplotlib pyarrow openpyxl tabulate

## 1. Импорты и список датасетов

В `DATASET_SLUGS` можно добавлять любые Kaggle slugs в формате `owner/dataset-name`.

Если какой-то slug не скачивается, ноутбук не падает, а записывает ошибку и идёт дальше.

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import kagglehub
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)

OUTPUT_DIR = Path('artifacts/run_05_multi_dataset_research')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Базовый список. Первый точно тот, который ты предложила.
# Остальные добавляй сюда вручную, когда находишь на Kaggle подходящий slug.
DATASET_SLUGS = [
    'oliviervha/crypto-news',

    # Примеры, куда вставлять следующие датасеты:
    # 'owner/dataset-name',
    # 'another-owner/another-dataset',
]

# Можно также создать рядом файл dataset_slugs.txt, где каждый slug на новой строке.
SLUG_FILE = Path('dataset_slugs.txt')
if SLUG_FILE.exists():
    extra_slugs = [x.strip() for x in SLUG_FILE.read_text(encoding='utf-8').splitlines() if x.strip() and not x.strip().startswith('#')]
    DATASET_SLUGS.extend(extra_slugs)

DATASET_SLUGS = list(dict.fromkeys(DATASET_SLUGS))
print('Datasets to check:')
for slug in DATASET_SLUGS:
    print('-', slug)

ASSET_KEYWORDS = {
    'BTC': ['btc', 'bitcoin', 'xbt'],
    'ETH': ['eth', 'ethereum', 'ether'],
    'SOL': ['sol', 'solana'],
    'BNB': ['bnb', 'binance coin', 'binance'],
    'XRP': ['xrp', 'ripple'],
    'ADA': ['ada', 'cardano'],
    'DOGE': ['doge', 'dogecoin'],
    'AVAX': ['avax', 'avalanche'],
    'MATIC': ['matic', 'polygon'],
    'DOT': ['dot', 'polkadot'],
    'LINK': ['link', 'chainlink'],
    'LTC': ['ltc', 'litecoin'],
}

YF_TICKERS = {
    'BTC': 'BTC-USD',
    'ETH': 'ETH-USD',
    'SOL': 'SOL-USD',
    'BNB': 'BNB-USD',
    'XRP': 'XRP-USD',
    'ADA': 'ADA-USD',
    'DOGE': 'DOGE-USD',
    'AVAX': 'AVAX-USD',
    'MATIC': 'MATIC-USD',
    'DOT': 'DOT-USD',
    'LINK': 'LINK-USD',
    'LTC': 'LTC-USD',
}

DATE_HINTS = ['date', 'time', 'timestamp', 'published', 'created', 'updated']
TEXT_HINTS = ['title', 'text', 'content', 'body', 'summary', 'description', 'headline', 'article', 'news']
ASSET_HINTS = ['symbol', 'ticker', 'coin', 'currency', 'asset', 'cryptocurrency', 'category']
SOURCE_HINTS = ['source', 'publisher', 'site', 'domain', 'author']
SUPPORTED_SUFFIXES = {'.csv', '.json', '.jsonl', '.parquet', '.xlsx', '.xls'}

print('Output:', OUTPUT_DIR.resolve())

## 2. Скачать все датасеты через `kagglehub`

In [ ]:
download_rows = []
dataset_paths = {}

for slug in DATASET_SLUGS:
    print('\nDownloading:', slug)
    try:
        path = kagglehub.dataset_download(slug)
        dataset_paths[slug] = Path(path)
        print('  OK:', path)
        download_rows.append({'dataset_slug': slug, 'status': 'ok', 'path': path, 'error': ''})
    except Exception as e:
        print('  ERROR:', repr(e))
        download_rows.append({'dataset_slug': slug, 'status': 'error', 'path': '', 'error': repr(e)})

downloads_df = pd.DataFrame(download_rows)
display(downloads_df)
downloads_df.to_csv(OUTPUT_DIR / 'downloads.csv', index=False)

## 3. Общие функции профилирования

In [ ]:
def file_size_mb(p: Path) -> float:
    return p.stat().st_size / 1024 / 1024

def read_table(path: Path, max_rows=None):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        for enc in ['utf-8', 'utf-8-sig', 'latin1']:
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False, nrows=max_rows)
            except Exception:
                pass
        return pd.read_csv(path, low_memory=False, nrows=max_rows)
    if suffix in ['.json', '.jsonl']:
        try:
            return pd.read_json(path, lines=True)
        except Exception:
            return pd.read_json(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path, nrows=max_rows)
    raise ValueError(f'Unsupported suffix: {suffix}')

def candidate_cols(df, hints):
    out = []
    for c in df.columns:
        name = str(c).lower()
        if any(h in name for h in hints):
            out.append(c)
    return out

def date_parse_rate(s, sample_size=5000):
    sample = s.dropna().astype(str).head(sample_size)
    if len(sample) == 0:
        return 0.0, None, None
    parsed = pd.to_datetime(sample, errors='coerce', utc=True)
    rate = parsed.notna().mean()
    if parsed.notna().any():
        return float(rate), parsed.min(), parsed.max()
    return float(rate), None, None

def text_score(s):
    sample = s.dropna().astype(str).head(5000)
    if len(sample) == 0:
        return {'non_null': 0, 'avg_len': 0, 'median_len': 0}
    lengths = sample.str.len()
    return {
        'non_null': int(s.notna().sum()),
        'avg_len': float(lengths.mean()),
        'median_len': float(lengths.median()),
    }

def count_asset_mentions(text_series):
    sample_text = text_series.fillna('').astype(str).str.lower()
    counts = {}
    for asset, words in ASSET_KEYWORDS.items():
        pattern = r'\b(' + '|'.join(re.escape(w.lower()) for w in words) + r')\b'
        counts[asset] = int(sample_text.str.contains(pattern, regex=True).sum())
    return counts

def normalize_events(df, date_col, text_cols, asset_col=None):
    out = pd.DataFrame(index=df.index)
    out['event_time'] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    out['event_date'] = out['event_time'].dt.tz_convert(None).dt.floor('D')

    text = pd.Series('', index=df.index, dtype='object')
    for c in text_cols:
        text = text + ' ' + df[c].fillna('').astype(str)
    out['text'] = text.str.strip()
    out['text_len'] = out['text'].str.len()

    if asset_col is not None and asset_col in df.columns:
        out['asset_raw'] = df[asset_col].fillna('').astype(str)
    else:
        out['asset_raw'] = ''

    lower_text = (out['text'] + ' ' + out['asset_raw']).fillna('').str.lower()
    matched = []
    for txt in lower_text:
        labels = []
        for asset, words in ASSET_KEYWORDS.items():
            pattern = r'\b(' + '|'.join(re.escape(w.lower()) for w in words) + r')\b'
            if re.search(pattern, txt):
                labels.append(asset)
        matched.append(labels)
    out['asset_list'] = matched
    out = out.dropna(subset=['event_time'])
    out = out[out['text_len'] > 10].copy()
    return out

def download_price(asset, start, end):
    yf_ticker = YF_TICKERS.get(asset)
    if yf_ticker is None:
        return pd.DataFrame()
    px = yf.download(yf_ticker, start=start, end=end, auto_adjust=False, progress=False)
    if px.empty:
        return pd.DataFrame()
    if isinstance(px.columns, pd.MultiIndex):
        px.columns = [c[0] for c in px.columns]
    px = px.reset_index()
    px.columns = [str(c).lower().replace(' ', '_') for c in px.columns]
    px = px.rename(columns={'date': 'date', 'adj_close': 'adj_close'})
    px['date'] = pd.to_datetime(px['date']).dt.floor('D')
    return px.sort_values('date').reset_index(drop=True)

def make_daily_text_features(events, asset):
    sub = events[events['asset_list'].apply(lambda xs: asset in xs)].copy()
    if sub.empty:
        return pd.DataFrame()
    daily = sub.groupby('event_date').agg(
        text_count=('text', 'size'),
        avg_text_len=('text_len', 'mean'),
        max_text_len=('text_len', 'max'),
    ).reset_index().rename(columns={'event_date': 'date'})
    daily['has_text'] = 1
    return daily

def evaluate_asset_candidate(dataset_slug, table_name, events, asset, horizons=(1, 3, 7)):
    daily_text = make_daily_text_features(events, asset)
    if daily_text.empty:
        return []
    start = (daily_text['date'].min() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
    end = (daily_text['date'].max() + pd.Timedelta(days=max(horizons) + 30)).strftime('%Y-%m-%d')
    px = download_price(asset, start, end)
    if px.empty:
        return []
    rows = []
    merged = px.merge(daily_text, on='date', how='left')
    for col in ['text_count', 'avg_text_len', 'max_text_len', 'has_text']:
        if col not in merged.columns:
            merged[col] = 0
    merged[['text_count', 'avg_text_len', 'max_text_len', 'has_text']] = merged[['text_count', 'avg_text_len', 'max_text_len', 'has_text']].fillna(0)
    for h in horizons:
        temp = merged.copy()
        temp[f'future_close_{h}d'] = temp['close'].shift(-h)
        temp[f'y_{h}d'] = (temp[f'future_close_{h}d'] > temp['close']).astype(float)
        temp = temp.dropna(subset=[f'future_close_{h}d'])
        active = temp[temp['text_count'] > 0]
        pos_share = active[f'y_{h}d'].mean() if len(active) else np.nan
        rows.append({
            'dataset_slug': dataset_slug,
            'table': table_name,
            'asset': asset,
            'yf_ticker': YF_TICKERS.get(asset),
            'horizon_days': h,
            'price_rows': int(len(px)),
            'news_dates': int(daily_text['date'].nunique()),
            'overlap_news_price_dates': int(temp[temp['text_count'] > 0]['date'].nunique()),
            'total_news_on_price_dates': int(temp['text_count'].sum()),
            'date_min': temp['date'].min(),
            'date_max': temp['date'].max(),
            'target_positive_share_on_news_dates': round(float(pos_share), 3) if pd.notna(pos_share) else np.nan,
        })
    return rows

## 4. Профилирование всех скачанных датасетов

In [ ]:
all_file_rows = []
all_table_profiles = []
all_asset_profiles = []
all_candidate_rows = []
best_tables = []
read_errors = []
normalized_events_by_key = {}

for slug, data_dir in dataset_paths.items():
    print('\n' + '=' * 100)
    print('DATASET:', slug)
    print('PATH:', data_dir)

    files = [p for p in data_dir.rglob('*') if p.is_file()]
    for p in files:
        all_file_rows.append({
            'dataset_slug': slug,
            'path': str(p),
            'name': p.name,
            'suffix': p.suffix.lower(),
            'size_mb': round(file_size_mb(p), 3),
        })

    for p in files:
        if p.suffix.lower() not in SUPPORTED_SUFFIXES:
            continue
        print('Reading:', p.name)
        try:
            df = read_table(p)
        except Exception as e:
            read_errors.append({'dataset_slug': slug, 'file': str(p), 'error': repr(e)})
            print('  read error:', repr(e))
            continue

        date_cols = candidate_cols(df, DATE_HINTS)
        text_cols = candidate_cols(df, TEXT_HINTS)
        asset_cols = candidate_cols(df, ASSET_HINTS)
        source_cols = candidate_cols(df, SOURCE_HINTS)

        date_info = []
        for c in date_cols:
            rate, dmin, dmax = date_parse_rate(df[c])
            date_info.append((c, rate, dmin, dmax))
        best_date = max(date_info, key=lambda x: x[1], default=(None, 0, None, None))

        text_info = []
        for c in text_cols:
            ts = text_score(df[c])
            text_info.append((c, ts['non_null'], ts['avg_len'], ts['median_len']))
        best_text = max(text_info, key=lambda x: (x[1], x[2]), default=(None, 0, 0, 0))

        combined_text = pd.Series('', index=df.index, dtype='object')
        for c in text_cols:
            combined_text = combined_text + ' ' + df[c].fillna('').astype(str)
        if asset_cols:
            combined_text = combined_text + ' ' + df[asset_cols[0]].fillna('').astype(str)
        mention_counts = count_asset_mentions(combined_text) if text_cols or asset_cols else {}

        rows = len(df)
        has_date = best_date[1] > 0.8
        has_text = best_text[0] is not None and best_text[2] >= 15
        total_mentions = sum(mention_counts.values())
        score = 0
        score += min(rows / 10000, 1) * 25
        score += 25 if has_date else 0
        score += 25 if has_text else 0
        score += min(total_mentions / max(rows, 1), 1) * 25

        profile = {
            'dataset_slug': slug,
            'table': p.name,
            'rows': rows,
            'cols': df.shape[1],
            'best_date_col': best_date[0],
            'date_parse_rate': round(best_date[1], 3),
            'date_min': best_date[2],
            'date_max': best_date[3],
            'best_text_col': best_text[0],
            'best_text_avg_len': round(best_text[2], 1),
            'text_cols': ', '.join(map(str, text_cols)),
            'asset_cols': ', '.join(map(str, asset_cols)),
            'source_cols': ', '.join(map(str, source_cols)),
            'asset_mentions_total': total_mentions,
            'BTC_mentions': mention_counts.get('BTC', 0),
            'ETH_mentions': mention_counts.get('ETH', 0),
            'SOL_mentions': mention_counts.get('SOL', 0),
            'suitability_score_raw_table': round(score, 1),
        }
        all_table_profiles.append(profile)

        if has_date and has_text:
            asset_col = asset_cols[0] if asset_cols else None
            events = normalize_events(df, best_date[0], text_cols, asset_col)
            key = (slug, p.name)
            normalized_events_by_key[key] = events

            for asset in ASSET_KEYWORDS:
                mask = events['asset_list'].apply(lambda xs: asset in xs)
                sub = events[mask].copy()
                if sub.empty:
                    continue
                daily_counts = sub.groupby('event_date').size()
                all_asset_profiles.append({
                    'dataset_slug': slug,
                    'table': p.name,
                    'asset': asset,
                    'n_events': int(len(sub)),
                    'n_dates': int(sub['event_date'].nunique()),
                    'date_min': sub['event_date'].min(),
                    'date_max': sub['event_date'].max(),
                    'avg_events_per_active_date': round(float(daily_counts.mean()), 2),
                    'median_events_per_active_date': round(float(daily_counts.median()), 2),
                })

            # Price overlap only for top assets by text count per table, to save time.
            table_asset_counts = []
            for asset in ASSET_KEYWORDS:
                n = int(events['asset_list'].apply(lambda xs: asset in xs).sum())
                if n > 0:
                    table_asset_counts.append((asset, n))
            table_asset_counts = sorted(table_asset_counts, key=lambda x: x[1], reverse=True)[:6]

            for asset, _ in table_asset_counts:
                all_candidate_rows.extend(evaluate_asset_candidate(slug, p.name, events, asset, horizons=(1, 3, 7)))

files_df = pd.DataFrame(all_file_rows).sort_values(['dataset_slug', 'size_mb'], ascending=[True, False])
table_profiles = pd.DataFrame(all_table_profiles).sort_values('suitability_score_raw_table', ascending=False) if all_table_profiles else pd.DataFrame()
asset_profiles = pd.DataFrame(all_asset_profiles).sort_values(['n_dates', 'n_events'], ascending=False) if all_asset_profiles else pd.DataFrame()
candidate_df = pd.DataFrame(all_candidate_rows)
errors_df = pd.DataFrame(read_errors)

if not candidate_df.empty:
    candidate_df['target_balance_score'] = 1 - (candidate_df['target_positive_share_on_news_dates'] - 0.5).abs().fillna(0.5) * 2
    candidate_df['suitability_score'] = (
        np.minimum(candidate_df['overlap_news_price_dates'] / 365, 1) * 35
        + np.minimum(candidate_df['total_news_on_price_dates'] / 5000, 1) * 35
        + candidate_df['target_balance_score'] * 20
        + np.minimum(candidate_df['news_dates'] / 365, 1) * 10
    ).round(1)
    candidate_df = candidate_df.sort_values('suitability_score', ascending=False)

display(table_profiles.head(30))
display(asset_profiles.head(30))
display(candidate_df.head(30) if not candidate_df.empty else candidate_df)

files_df.to_csv(OUTPUT_DIR / 'all_dataset_files.csv', index=False)
table_profiles.to_csv(OUTPUT_DIR / 'all_table_profiles.csv', index=False)
asset_profiles.to_csv(OUTPUT_DIR / 'all_asset_text_coverage.csv', index=False)
candidate_df.to_csv(OUTPUT_DIR / 'all_price_overlap_candidates.csv', index=False)
errors_df.to_csv(OUTPUT_DIR / 'read_errors.csv', index=False)

print('Saved profiling artifacts to:', OUTPUT_DIR)

## 5. Leaderboard датасетов

Смотрим лучшие пары `dataset / table / asset / horizon`. Это кандидаты для полного RUN 05.

In [ ]:
if candidate_df.empty:
    leaderboard = pd.DataFrame()
    print('Нет кандидатов. Нужно проверить date/text columns или добавить другие slugs.')
else:
    leaderboard = candidate_df.copy()
    leaderboard = leaderboard.sort_values('suitability_score', ascending=False)
    cols = [
        'dataset_slug', 'table', 'asset', 'horizon_days', 'suitability_score',
        'overlap_news_price_dates', 'total_news_on_price_dates',
        'target_positive_share_on_news_dates', 'date_min', 'date_max'
    ]
    display(leaderboard[cols].head(25))
    leaderboard[cols].head(50).to_csv(OUTPUT_DIR / 'dataset_leaderboard_top50.csv', index=False)

    ax = leaderboard.head(20).plot(x='asset', y='suitability_score', kind='bar', legend=False, figsize=(11, 4))
    ax.set_title('Top dataset/asset/horizon candidates')
    ax.set_xlabel('asset')
    ax.set_ylabel('suitability score')
    plt.tight_layout()
    plt.show()

## 6. Short-list для RUN 05

Фильтр: минимум 30 пересечений news+price dates и минимум 100 новостей на price dates. Если таких нет, берём top-10 по score.

In [ ]:
if candidate_df.empty:
    shortlist = pd.DataFrame()
else:
    shortlist = candidate_df[
        (candidate_df['overlap_news_price_dates'] >= 30)
        & (candidate_df['total_news_on_price_dates'] >= 100)
    ].copy()
    if shortlist.empty:
        shortlist = candidate_df.head(10).copy()
    else:
        shortlist = shortlist.head(15).copy()

display(shortlist)
shortlist.to_csv(OUTPUT_DIR / 'shortlist_for_run05.csv', index=False)

if not shortlist.empty:
    print('Best candidate:')
    display(shortlist.iloc[[0]])

## 7. Собрать sample dataset для лучшего кандидата

Это ещё не полный pipeline. Это проверочный файл, чтобы убедиться, что выбранный dataset реально можно совместить с ценами.

In [ ]:
if shortlist.empty:
    print('Shortlist empty. Добавь ещё dataset slugs и перезапусти notebook.')
else:
    best = shortlist.iloc[0]
    BEST_SLUG = best['dataset_slug']
    BEST_TABLE = best['table']
    BEST_ASSET = best['asset']
    BEST_H = int(best['horizon_days'])

    key = (BEST_SLUG, BEST_TABLE)
    events = normalized_events_by_key[key]
    daily_text = make_daily_text_features(events, BEST_ASSET)

    start = (daily_text['date'].min() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
    end = (daily_text['date'].max() + pd.Timedelta(days=BEST_H + 30)).strftime('%Y-%m-%d')
    px = download_price(BEST_ASSET, start, end)

    sample = px.merge(daily_text, on='date', how='left')
    for col in ['text_count', 'avg_text_len', 'max_text_len', 'has_text']:
        if col not in sample.columns:
            sample[col] = 0
    sample[['text_count', 'avg_text_len', 'max_text_len', 'has_text']] = sample[['text_count', 'avg_text_len', 'max_text_len', 'has_text']].fillna(0)
    sample[f'future_close_{BEST_H}d'] = sample['close'].shift(-BEST_H)
    sample['y'] = (sample[f'future_close_{BEST_H}d'] > sample['close']).astype(float)
    sample = sample.dropna(subset=[f'future_close_{BEST_H}d']).reset_index(drop=True)

    safe_slug = BEST_SLUG.replace('/', '__')
    sample_path = OUTPUT_DIR / f'sample_{safe_slug}_{BEST_TABLE}_{BEST_ASSET}_{BEST_H}d.csv'
    sample.to_csv(sample_path, index=False)

    print('BEST_SLUG:', BEST_SLUG)
    print('BEST_TABLE:', BEST_TABLE)
    print('BEST_ASSET:', BEST_ASSET)
    print('BEST_HORIZON_DAYS:', BEST_H)
    print('sample shape:', sample.shape)
    print('saved:', sample_path)
    display(sample.head())
    display(sample[['date', 'close', 'volume', 'text_count', 'has_text', 'y']].tail())

## 8. Сохранить summary

После запуска пришли мне:

- `all_table_profiles.csv`
- `all_asset_text_coverage.csv`
- `all_price_overlap_candidates.csv`
- `dataset_leaderboard_top50.csv`
- `shortlist_for_run05.csv`
- `multi_dataset_research_notes.md`

По ним выберем лучший датасет и напишем RUN 05 pipeline.

In [ ]:
notes = []
notes.append('# RUN 05 multi-dataset research notes\n')
notes.append('## Checked datasets\n')
for slug in DATASET_SLUGS:
    notes.append(f'- `{slug}`')

if not downloads_df.empty:
    notes.append('\n## Downloads\n')
    notes.append(downloads_df.to_markdown(index=False))

if not table_profiles.empty:
    notes.append('\n## Best tables\n')
    notes.append(table_profiles.head(10).to_markdown(index=False))

if not candidate_df.empty:
    notes.append('\n## Top candidates\n')
    cols = ['dataset_slug', 'table', 'asset', 'horizon_days', 'suitability_score', 'overlap_news_price_dates', 'total_news_on_price_dates', 'target_positive_share_on_news_dates']
    notes.append(candidate_df[cols].head(15).to_markdown(index=False))

if not shortlist.empty:
    notes.append('\n## Shortlist\n')
    notes.append(shortlist.to_markdown(index=False))

notes_path = OUTPUT_DIR / 'multi_dataset_research_notes.md'
notes_path.write_text('\n'.join(notes), encoding='utf-8')

print('Saved artifacts:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name)